In [10]:
# Cell 1: Import Libraries and Setup
import os
import random
import shutil
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize, remove_small_objects
from scipy.spatial.distance import cdist
import glob
from scipy import ndimage
import pickle
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import seaborn as sns
from skimage.feature import hog, local_binary_pattern

import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  
print("All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"OpenCV version: {cv2.__version__}")


All libraries imported successfully!
TensorFlow version: 2.19.0
OpenCV version: 4.12.0


In [11]:
!rm -r "/kaggle/working/Landrace_Pietra_renamed"


In [12]:
src_dir = "/kaggle/input/pig-identification-data-001/Duroc-Pietra/Duroc-Pietra"
dst_dir = "/kaggle/working/Landrace_Pietra_renamed"

count = 0
for root, dirs, files in os.walk(src_dir):
    for filename in files:
        if filename.endswith(".JPG"):  # Case-sensitive match
            # Change extension to lowercase
            new_filename = os.path.splitext(filename)[0] + ".jpg"

            # Preserve folder structure
            rel_path = os.path.relpath(root, src_dir)
            dst_subdir = os.path.join(dst_dir, rel_path)
            os.makedirs(dst_subdir, exist_ok=True)

            src_path = os.path.join(root, filename)
            dst_path = os.path.join(dst_subdir, new_filename)

            shutil.copy2(src_path, dst_path)
            count += 1

print(f"Renamed and copied {count} '.JPG' images to '.jpg' in: {dst_dir}")


Renamed and copied 200 '.JPG' images to '.jpg' in: /kaggle/working/Landrace_Pietra_renamed


In [13]:
src_dir = "/kaggle/input/pig-identification-data-001/LandracePietra65kg/Landrace+Pietra65kg"
dst_dir = "/kaggle/working/Landrace_Pietra_renamed"

count = 0
for root, dirs, files in os.walk(src_dir):
    for filename in files:
        if filename.endswith(".JPG"):  # Case-sensitive match
            # Change extension to lowercase
            new_filename = os.path.splitext(filename)[0] + ".jpg"

            # Preserve folder structure
            rel_path = os.path.relpath(root, src_dir)
            dst_subdir = os.path.join(dst_dir, rel_path)
            os.makedirs(dst_subdir, exist_ok=True)

            src_path = os.path.join(root, filename)
            dst_path = os.path.join(dst_subdir, new_filename)

            shutil.copy2(src_path, dst_path)
            count += 1

print(f"Renamed and copied {count} '.JPG' images to '.jpg' in: {dst_dir}")


Renamed and copied 360 '.JPG' images to '.jpg' in: /kaggle/working/Landrace_Pietra_renamed


In [14]:
count=0
for dirs in os.listdir(dst_dir):
    sub_folder=os.path.join(dst_dir, dirs)
    for imgs_folder in os.listdir(sub_folder):
        count+=1
print('total images are:', count)

total images are: 560


In [15]:
src_dir = "/kaggle/input/pig-identification-data-001/LandracePietra90kg/Landrace+Pietra90kg"
dst_dir = "/kaggle/working/Landrace_Pietra_renamed"

count = 0
for root, dirs, files in os.walk(src_dir):
    for filename in files:
        if filename.endswith(".JPG"):  # Case-sensitive match
            # Change extension to lowercase
            new_filename = os.path.splitext(filename)[0] + ".jpg"

            # Preserve folder structure
            rel_path = os.path.relpath(root, src_dir)
            dst_subdir = os.path.join(dst_dir, rel_path)
            os.makedirs(dst_subdir, exist_ok=True)

            src_path = os.path.join(root, filename)
            dst_path = os.path.join(dst_subdir, new_filename)

            shutil.copy2(src_path, dst_path)
            count += 1

print(f"Renamed and copied {count} '.JPG' images to '.jpg' in: {dst_dir}")


Renamed and copied 240 '.JPG' images to '.jpg' in: /kaggle/working/Landrace_Pietra_renamed


In [16]:
count=0
for dirs in os.listdir(dst_dir):
    sub_folder=os.path.join(dst_dir, dirs)
    for imgs_folder in os.listdir(sub_folder):
        count+=1
print('total images are:', count)

total images are: 800


In [17]:
def split_dataset(source_dir, output_dir, train_ratio=0.8, seed=42):
    random.seed(seed)

    train_dir = os.path.join(output_dir, 'training')
    test_dir = os.path.join(output_dir, 'testing')

    if os.path.exists(train_dir):
        shutil.rmtree(train_dir)
    os.makedirs(train_dir)

    if os.path.exists(test_dir):
        shutil.rmtree(test_dir)
    os.makedirs(test_dir)

    stats = {'total_pigs': 0, 'total_images': 0, 'training_images': 0, 'testing_images': 0}

    print(f"Looking for pig folders in: {source_dir}")

    for pig_folder in os.listdir(source_dir):
        pig_path = os.path.join(source_dir, pig_folder)
        if not os.path.isdir(pig_path):
            continue

        stats['total_pigs'] += 1
        pig_id = pig_folder
        print(f"Processing pig {pig_id}...")

        images = [f for f in os.listdir(pig_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if not images:
            print(f"  - No images found for pig {pig_id}")
            continue

        stats['total_images'] += len(images)
        random.shuffle(images)

        split_index = round(len(images) * train_ratio)
        training_images = images[:split_index]
        testing_images = images[split_index:]

        stats['training_images'] += len(training_images)
        stats['testing_images'] += len(testing_images)

        for img in training_images:
            src = os.path.join(pig_path, img)
            dest_filename = f"pig_{pig_id}_{img}"
            shutil.copy2(src, os.path.join(train_dir, dest_filename))

        for img in testing_images:
            src = os.path.join(pig_path, img)
            dest_filename = f"pig_{pig_id}_{img}"
            shutil.copy2(src, os.path.join(test_dir, dest_filename))

        print(f"  - {len(training_images)} images in training, {len(testing_images)} images in testing")

    assert stats['training_images'] + stats['testing_images'] == stats['total_images'], "Mismatch in total images!"

    print("\nDataset Split Statistics:")
    print(f"Total pigs: {stats['total_pigs']}")
    print(f"Total images: {stats['total_images']}")
    print(f"Training images: {stats['training_images']} ({stats['training_images']/stats['total_images']*100:.1f}%)")
    print(f"Testing images: {stats['testing_images']} ({stats['testing_images']/stats['total_images']*100:.1f}%)")

    return train_dir, test_dir

print("Dataset splitting function defined!")

Dataset splitting function defined!


In [18]:
# Configure paths
SOURCE_DIR = "/kaggle/working/Landrace_Pietra_renamed"
OUTPUT_DIR = "/kaggle/working/pig_dataset_split"

# Split the dataset
train_dir, test_dir = split_dataset(SOURCE_DIR, OUTPUT_DIR)
print(f"\nTraining set path: {train_dir}")
print(f"Testing set path: {test_dir}")

Looking for pig folders in: /kaggle/working/Landrace_Pietra_renamed
Processing pig 44052...
  - 32 images in training, 8 images in testing
Processing pig 351...
  - 32 images in training, 8 images in testing
Processing pig 15169...
  - 32 images in training, 8 images in testing
Processing pig 354...
  - 32 images in training, 8 images in testing
Processing pig 789...
  - 32 images in training, 8 images in testing
Processing pig 772...
  - 32 images in training, 8 images in testing
Processing pig 44014...
  - 32 images in training, 8 images in testing
Processing pig 13885...
  - 32 images in training, 8 images in testing
Processing pig 44015...
  - 32 images in training, 8 images in testing
Processing pig 14756...
  - 32 images in training, 8 images in testing
Processing pig 14760...
  - 32 images in training, 8 images in testing
Processing pig 13865...
  - 32 images in training, 8 images in testing
Processing pig 768...
  - 32 images in training, 8 images in testing
Processing pig 0536

In [ ]:
import os
from PIL import Image, ImageFilter, ImageEnhance
import numpy as np

class DataAugmentor:
    """Apply augmentations: noise, blur, brightness variations"""
    
    @staticmethod
    def add_gaussian_noise(img, std=25):
        """Add Gaussian noise"""
        img_array = np.array(img).astype(float)
        noise = np.random.normal(0, std, img_array.shape)
        noisy = np.clip(img_array + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(noisy)
    
    @staticmethod
    def add_blur(img, radius=3):
        """Apply Gaussian blur"""
        return img.filter(ImageFilter.GaussianBlur(radius=radius))
    
    @staticmethod
    def decrease_brightness(img, factor=0.7):
        """Decrease brightness (-30%)"""
        enhancer = ImageEnhance.Brightness(img)
        return enhancer.enhance(factor)
    
    @staticmethod
    def increase_brightness(img, factor=1.3):
        """Increase brightness (+30%)"""
        enhancer = ImageEnhance.Brightness(img)
        return enhancer.enhance(factor)

def create_augmented_dataset(input_dir, output_dir):
    """
    Create augmented dataset: for each image, creates 5 versions
    (original, noise, blur, dark, bright)
    
    Args:
        input_dir: Directory with original images
        output_dir: Directory to save augmented images
    
    Returns:
        Number of images created
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all images
    image_files = [f for f in os.listdir(input_dir) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    print(f"\nAugmenting {len(image_files)} images...")
    print(f"Output directory: {output_dir}")
    
    total_created = 0
    
    for idx, img_filename in enumerate(image_files):
        if (idx + 1) % 50 == 0:
            print(f"  Processed {idx + 1}/{len(image_files)} images...")
        
        # Load image
        img_path = os.path.join(input_dir, img_filename)
        img = Image.open(img_path).convert('RGB')
        
        # Get base name without extension
        base_name = os.path.splitext(img_filename)[0]
        ext = os.path.splitext(img_filename)[1]
        
        # 1. Save original
        img.save(os.path.join(output_dir, f"{base_name}_original{ext}"))
        total_created += 1
        
        # 2. Gaussian noise version
        noisy = DataAugmentor.add_gaussian_noise(img, std=25)
        noisy.save(os.path.join(output_dir, f"{base_name}_noise{ext}"))
        total_created += 1
        
        # 3. Blur version
        blurred = DataAugmentor.add_blur(img, radius=3)
        blurred.save(os.path.join(output_dir, f"{base_name}_blur{ext}"))
        total_created += 1
        
        # 4. Dark version (-30% brightness)
        dark = DataAugmentor.decrease_brightness(img, factor=0.7)
        dark.save(os.path.join(output_dir, f"{base_name}_dark{ext}"))
        total_created += 1
        
        # 5. Bright version (+30% brightness)
        bright = DataAugmentor.increase_brightness(img, factor=1.3)
        bright.save(os.path.join(output_dir, f"{base_name}_bright{ext}"))
        total_created += 1
    
    print(f"\n✅ Created {total_created} augmented images")
    print(f"   Original: {len(image_files)}")
    print(f"   Augmented: {total_created} (5x multiplication)")
    
    return total_created

# ============================================================================
# RUN AUGMENTATION
# ============================================================================

# Your directories
train_dir = "/kaggle/working/pig_dataset_split/training"
test_dir = "/kaggle/working/pig_dataset_split/testing"

# Output directories
augmented_train_dir = "/kaggle/working/augmented_training"
augmented_test_dir = "/kaggle/working/augmented_testing_combined"

print("="*80)
print("CREATING AUGMENTED DATASETS")
print("="*80)

# Augment training data
print("\n1. TRAINING DATA:")
print("-"*80)
train_count = create_augmented_dataset(train_dir, augmented_train_dir)

# Augment test data
print("\n2. TEST DATA:")
print("-"*80)
test_count = create_augmented_dataset(test_dir, augmented_test_dir)

print("\n" + "="*80)
print("AUGMENTATION COMPLETE!")
print("="*80)
print(f"\nTraining: {train_dir}")
print(f"       → {augmented_train_dir} ({train_count} images)")
print(f"\nTesting:  {test_dir}")
print(f"       → {augmented_test_dir} ({test_count} images)")
print("\n" + "="*80)
print("NEXT STEPS:")
print("="*80)
print("1. Use augmented_train_dir to train your models (SVM, MobileNet)")
print("2. Use augmented_test_dir to test robustness")
print("3. Compare models trained on original vs augmented data")
print("="*80)

CREATING AUGMENTED DATASETS

1. TRAINING DATA:
--------------------------------------------------------------------------------

Augmenting 640 images...
Output directory: /kaggle/working/augmented_training
  Processed 50/640 images...
  Processed 100/640 images...
  Processed 150/640 images...
  Processed 200/640 images...
  Processed 250/640 images...
  Processed 300/640 images...
  Processed 350/640 images...
  Processed 400/640 images...


In [ ]:
!pip install timm

In [ ]:
import timm

In [15]:
# ============================================================================
# FULL PIPELINE: TRAIN / VAL / TEST WITH MOBILE BENCHMARKING
# ============================================================================

import os
import cv2
import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================================================================
# MOBILE SIMULATION SETUP
# ============================================================================
device = "cpu"
torch.set_num_threads(4)
os.makedirs("checkpoints", exist_ok=True)

print("="*80)
print("📱 MOBILE SIMULATION MODE")
print(f"Device           : {device}")
print(f"CPU threads used : {torch.get_num_threads()}")
print("="*80)

# ============================================================================
# DATA TRANSFORMS
# ============================================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# ============================================================================
# IMAGE PROCESSING
# ============================================================================
def process_image_for_mobilenet(image_path, train=True):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    transform = train_transform if train else val_test_transform
    return transform(img)

# ============================================================================
# DATASET
# ============================================================================
import os
import re
import shutil
import torch
from torch.utils.data import Dataset


class EarDataset(Dataset):
    """
    Dataset for images named like:
    pig_[pig_id]_IMG_[img_id]_[augmentation].jpg

    - Label = pig_id
    - Optionally organizes test images into augmentation folders
    """
    def __init__(self, root_dir, train=True, organize_test_by_aug=True):
        self.root_dir = root_dir
        self.train = train
        self.images = []
        self.labels = []
        self.augmentations = []

        self.classes = []
        self.class_to_idx = {}
        self.idx_to_class = {}

        pattern = re.compile(r"pig_(\d+)_IMG_(\d+)_([a-zA-Z0-9]+)", re.IGNORECASE)

        pig_id_set = set()
        aug_set = set()

        if not train and organize_test_by_aug:
            print("🗂️  Organizing TEST images by augmentation type...")
            self._organize_by_augmentation(pattern)

        for fname in sorted(os.listdir(root_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            match = pattern.search(fname)
            if match is None:
                print(f"⚠️ Skipping malformed filename: {fname}")
                continue

            pig_id, img_id, aug = match.groups()

            self.images.append(os.path.join(root_dir, fname))
            self.labels.append(pig_id)
            self.augmentations.append(aug)
            pig_id_set.add(pig_id)
            aug_set.add(aug)

        self.classes = sorted(list(pig_id_set))
        self.class_to_idx = {c:i for i, c in enumerate(self.classes)}
        self.idx_to_class = {i:c for c, i in self.class_to_idx.items()}
        self.labels = [self.class_to_idx[lbl] for lbl in self.labels]

        print(f"✅ Total images loaded     : {len(self.images)}")
        print(f"🏷️  Unique pig_id classes  : {len(self.classes)}")
        print(f"🧪 Augmentation types     : {sorted(list(aug_set))}")

    def _organize_by_augmentation(self, pattern):
        for fname in os.listdir(self.root_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            match = pattern.search(fname)
            if match is None:
                continue

            pig_id, img_id, aug = match.groups()
            aug_dir = os.path.join(self.root_dir, aug)
            os.makedirs(aug_dir, exist_ok=True)
            src = os.path.join(self.root_dir, fname)
            dst = os.path.join(aug_dir, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
        print("✅ Test images organized by augmentation")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = process_image_for_mobilenet(self.images[idx], self.train)
        if img is None:
            img = torch.zeros(3, 224, 224)
        label = self.labels[idx]
        return img, label

# ============================================================================
# PATHS
# ============================================================================
TRAIN_ROOT = "/kaggle/working/augmented_training"
TEST_ROOT  = "/kaggle/working/augmented_testing_combined"

print("\n📂 DATASET PATHS")
print(f"Train root : {TRAIN_ROOT}")
print(f"Test root  : {TEST_ROOT}")

# ============================================================================
# LOAD & SPLIT TRAIN DATA
# ============================================================================
full_train_dataset = EarDataset(TRAIN_ROOT, train=True)
num_classes = len(full_train_dataset.classes)

val_ratio = 0.15
total_size = len(full_train_dataset)
val_size = int(val_ratio * total_size)
train_size = total_size - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset.dataset.train = True
val_dataset.dataset.train = False

print("\n📊 DATA SPLIT")
print(f"Total train images : {total_size}")
print(f"Train set          : {train_size} (85%)")
print(f"Validation set     : {val_size} (15%)")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False)

# ============================================================================
# TEST DATA
# ============================================================================
test_dataset = EarDataset(TEST_ROOT, train=False)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

print(f"Test set           : {len(test_dataset)} images")

# ============================================================================
# MODEL
# ============================================================================

model = timm.create_model(
    'efficientnet_lite0',
    pretrained=True,
    num_classes=num_classes
)

# Freeze backbone
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

model.to(device)

# Optimizer
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)
print("\n🧠 MODEL")
print("Architecture : MobileViT-S")
print(f"Classes      : {num_classes}")

# ============================================================================
# TRAINING UTILITIES
# ============================================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

def validate_simple(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device))
            y_pred.append(out.argmax(1).item())
            y_true.append(y.item())
    return accuracy_score(y_true, y_pred)

# ============================================================================
# TRAINING LOOP
# ============================================================================
best_val_acc = 0.0
patience = 4
epochs_no_improve = 0

print("\n🚀 TRAINING STARTED")
for epoch in range(1, 8):
    model.train()
    epoch_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    val_acc = validate_simple(model, val_loader)

    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {epoch_loss/len(train_loader):.4f} | "
          f"Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "checkpoints/best_model.pth")
        print("✅ New best model saved")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement ({epochs_no_improve}/{patience})")

    if epochs_no_improve >= patience:
        print("🛑 Early stopping triggered")
        break

# ============================================================================
# FINAL TEST EVALUATION (HELD-OUT)
# ============================================================================
print("\n" + "="*80)
print("🧪 FINAL TEST SET EVALUATION (HELD-OUT)")
print("="*80)

model.load_state_dict(torch.load("checkpoints/best_model.pth"))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        y_pred.append(out.argmax(1).item())
        y_true.append(y.item())

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("\n📊 TEST METRICS")
print(f"Accuracy  : {acc*100:.2f}%")
print(f"Precision : {prec*100:.2f}%")
print(f"Recall    : {rec*100:.2f}%")
print(f"F1-score  : {f1*100:.2f}%")

print("\n✅ PIPELINE COMPLETE")


📱 MOBILE SIMULATION MODE
Device           : cpu
CPU threads used : 4

📂 DATASET PATHS
Train root : /kaggle/working/augmented_training
Test root  : /kaggle/working/augmented_testing_combined
✅ Total images loaded     : 3200
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['blur', 'bright', 'dark', 'noise', 'original']

📊 DATA SPLIT
Total train images : 3200
Train set          : 2720 (85%)
Validation set     : 480 (15%)
🗂️  Organizing TEST images by augmentation type...
✅ Test images organized by augmentation
✅ Total images loaded     : 800
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['blur', 'bright', 'dark', 'noise', 'original']
Test set           : 800 images

🧠 MODEL
Architecture : MobileViT-S
Classes      : 20

🚀 TRAINING STARTED
Epoch 01 | Train Loss: 1.2801 | Val Acc: 99.17%
✅ New best model saved
Epoch 02 | Train Loss: 0.1683 | Val Acc: 100.00%
✅ New best model saved
Epoch 03 | Train Loss: 0.0749 | Val Acc: 100.00%
⏳ No improvement (1/4)
Epoch 04 | Train

In [34]:
performance_dict={"Models":['MobileViT-S'], "Augmentations": ['Combined Perturbations'], "Accuracy": [98.50],'Precision':[98.58], "Recall": [98.50], "F1-score":[98.46] }

In [26]:
performance_dict.keys()

dict_keys(['Models', 'Augmentations', 'Accuracy', 'Precision', 'Recall', 'F1-score'])

In [12]:
!pip install timm

In [13]:
import timm

In [ ]:
# ============================================================================
# FULL PIPELINE: TRAIN / VAL / TEST WITH MOBILE BENCHMARKING
# ============================================================================

import os
import cv2
import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================================================================
# MOBILE SIMULATION SETUP
# ============================================================================
device = "cpu"
torch.set_num_threads(4)
os.makedirs("checkpoints", exist_ok=True)

print("="*80)
print("📱 MOBILE SIMULATION MODE")
print(f"Device           : {device}")
print(f"CPU threads used : {torch.get_num_threads()}")
print("="*80)

# ============================================================================
# DATA TRANSFORMS
# ============================================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# ============================================================================
# IMAGE PROCESSING
# ============================================================================
def process_image_for_mobilenet(image_path, train=True):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    transform = train_transform if train else val_test_transform
    return transform(img)

# ============================================================================
# DATASET
# ============================================================================
import os
import re
import shutil
import torch
from torch.utils.data import Dataset


class EarDataset(Dataset):
    """
    Dataset for images named like:
    pig_[pig_id]_IMG_[img_id]_[augmentation].jpg

    - Label = pig_id
    - Optionally organizes test images into augmentation folders
    """
    def __init__(self, root_dir, train=True):
        self.root_dir = root_dir
        self.train = train
        self.images = []
        self.labels = []
        self.augmentations = []

        self.classes = []
        self.class_to_idx = {}
        self.idx_to_class = {}

        pattern = re.compile(r"pig_(\d+)_IMG_(\d+)", re.IGNORECASE)

        pig_id_set = set()
       # aug_set = set()

        # if not train :
        #     print("🗂️  Organizing TEST images by augmentation type...")
        #     self._organize_by_augmentation(pattern)

        for fname in sorted(os.listdir(root_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            match = pattern.search(fname)
            if match is None:
                print(f"⚠️ Skipping malformed filename: {fname}")
                continue

            pig_id, img_id = match.groups()

            self.images.append(os.path.join(root_dir, fname))
            self.labels.append(pig_id)
            #self.augmentations.append(aug)
            pig_id_set.add(pig_id)
            #aug_set.add(aug)

        self.classes = sorted(list(pig_id_set))
        self.class_to_idx = {c:i for i, c in enumerate(self.classes)}
        self.idx_to_class = {i:c for c, i in self.class_to_idx.items()}
        self.labels = [self.class_to_idx[lbl] for lbl in self.labels]

        print(f"✅ Total images loaded     : {len(self.images)}")
        print(f"🏷️  Unique pig_id classes  : {len(self.classes)}")
        #print(f"🧪 Augmentation types     : {sorted(list(aug_set))}")

    # def _organize_by_augmentation(self, pattern):
    #     for fname in os.listdir(self.root_dir):
    #         if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
    #             continue
    #         match = pattern.search(fname)
    #         if match is None:
    #             continue

    #         pig_id, img_id, aug = match.groups()
    #         aug_dir = os.path.join(self.root_dir, aug)
    #         os.makedirs(aug_dir, exist_ok=True)
    #         src = os.path.join(self.root_dir, fname)
    #         dst = os.path.join(aug_dir, fname)
    #         if not os.path.exists(dst):
    #             shutil.copy2(src, dst)
    #     print("✅ Test images organized by augmentation")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = process_image_for_mobilenet(self.images[idx], self.train)
        if img is None:
            img = torch.zeros(3, 224, 224)
        label = self.labels[idx]
        return img, label

# ============================================================================
# PATHS
# ============================================================================
TRAIN_ROOT = "/kaggle/working/pig_dataset_split/training"
TEST_ROOT  = "/kaggle/working/pig_dataset_split/training"

print("\n📂 DATASET PATHS")
print(f"Train root : {TRAIN_ROOT}")
print(f"Test root  : {TEST_ROOT}")

# ============================================================================
# LOAD & SPLIT TRAIN DATA
# ============================================================================
full_train_dataset = EarDataset(TRAIN_ROOT, train=True)
num_classes = len(full_train_dataset.classes)

val_ratio = 0.15
total_size = len(full_train_dataset)
val_size = int(val_ratio * total_size)
train_size = total_size - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset.dataset.train = True
val_dataset.dataset.train = False

print("\n📊 DATA SPLIT")
print(f"Total train images : {total_size}")
print(f"Train set          : {train_size} (85%)")
print(f"Validation set     : {val_size} (15%)")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False)

# ============================================================================
# TEST DATA
# ============================================================================
test_dataset = EarDataset(TEST_ROOT, train=False)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

print(f"Test set           : {len(test_dataset)} images")

# ============================================================================
# MODEL
# ============================================================================

model = timm.create_model(
    'efficientnet_lite0',
    pretrained=True,
    num_classes=num_classes
)

# Freeze backbone
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

model.to(device)

# Optimizer
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)
print("\n🧠 MODEL")
print("Architecture : MobileViT-S")
print(f"Classes      : {num_classes}")

# ============================================================================
# TRAINING UTILITIES
# ============================================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

def validate_simple(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device))
            y_pred.append(out.argmax(1).item())
            y_true.append(y.item())
    return accuracy_score(y_true, y_pred)

# ============================================================================
# TRAINING LOOP
# ============================================================================
best_val_acc = 0.0
patience = 4
epochs_no_improve = 0

print("\n🚀 TRAINING STARTED")
for epoch in range(1, 8):
    model.train()
    epoch_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    val_acc = validate_simple(model, val_loader)

    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {epoch_loss/len(train_loader):.4f} | "
          f"Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "checkpoints/best_model.pth")
        print("✅ New best model saved")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement ({epochs_no_improve}/{patience})")

    if epochs_no_improve >= patience:
        print("🛑 Early stopping triggered")
        break

# ============================================================================
# FINAL TEST EVALUATION (HELD-OUT)
# ============================================================================
print("\n" + "="*80)
print("🧪 FINAL TEST SET EVALUATION (HELD-OUT)")
print("="*80)

model.load_state_dict(torch.load("checkpoints/best_model.pth"))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        y_pred.append(out.argmax(1).item())
        y_true.append(y.item())

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("\n📊 TEST METRICS")
print(f"Accuracy  : {acc*100:.2f}%")
print(f"Precision : {prec*100:.2f}%")
print(f"Recall    : {rec*100:.2f}%")
print(f"F1-score  : {f1*100:.2f}%")

print("\n✅ PIPELINE COMPLETE")


In [ ]:
# ============================================================================
# FULL PIPELINE: TRAIN / VAL / TEST WITH MOBILENETV3-SMALL (MOBILE-FRIENDLY)
# ============================================================================

import os
import cv2
import re
import time
import torch
import shutil
import numpy as np
import torch.nn as nn
import torch.optim as optim

from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================================================================
# MOBILE SIMULATION SETUP
# ============================================================================
device = "cpu"
torch.set_num_threads(4)
os.makedirs("checkpoints", exist_ok=True)

print("=" * 80)
print("📱 MOBILE SIMULATION MODE")
print(f"Device           : {device}")
print(f"CPU threads used : {torch.get_num_threads()}")
print("=" * 80)

# ============================================================================
# DATA TRANSFORMS
# ============================================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================================================
# IMAGE PROCESSING
# ============================================================================
def process_image(image_path, train=True):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    transform = train_transform if train else val_test_transform
    return transform(img)

# ============================================================================
# DATASET
# ============================================================================
class EarDataset(Dataset):
    """
    Dataset for images named:
    pig_[pig_id]_IMG_[img_id].jpg
    """

    def __init__(self, root_dir, train=True):
        self.root_dir = root_dir
        self.train = train
        self.images = []
        self.labels = []

        pattern = re.compile(r"pig_(\d+)_IMG_(\d+)", re.IGNORECASE)
        pig_ids = set()

        for fname in sorted(os.listdir(root_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            match = pattern.search(fname)
            if match is None:
                print(f"⚠️ Skipping malformed filename: {fname}")
                continue

            pig_id, _ = match.groups()
            self.images.append(os.path.join(root_dir, fname))
            self.labels.append(pig_id)
            pig_ids.add(pig_id)

        self.classes = sorted(list(pig_ids))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.labels = [self.class_to_idx[l] for l in self.labels]

        print(f"✅ Loaded images : {len(self.images)}")
        print(f"🏷️ Classes      : {len(self.classes)}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = process_image(self.images[idx], self.train)
        if img is None:
            img = torch.zeros(3, 224, 224)
        return img, self.labels[idx]

# ============================================================================
# PATHS
# ============================================================================
TRAIN_ROOT = "/kaggle/working/pig_dataset_split/training"
TEST_ROOT  = "/kaggle/working/pig_dataset_split/training"

print("\n📂 DATASET PATHS")
print(f"Train root : {TRAIN_ROOT}")
print(f"Test root  : {TEST_ROOT}")

# ============================================================================
# LOAD & SPLIT TRAIN DATA
# ============================================================================
full_train_dataset = EarDataset(TRAIN_ROOT, train=True)
num_classes = len(full_train_dataset.classes)

val_ratio = 0.15
val_size = int(len(full_train_dataset) * val_ratio)
train_size = len(full_train_dataset) - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset.dataset.train = True
val_dataset.dataset.train = False

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False)

# ============================================================================
# TEST DATA
# ============================================================================
test_dataset = EarDataset(TEST_ROOT, train=False)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

# ============================================================================
# MODEL (MobileNetV3-Small)
# ============================================================================
model = mobilenet_v3_small(
    weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1
)

# Replace classifier
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, num_classes)

# Freeze backbone
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

model.to(device)

print("\n🧠 MODEL")
print("Architecture : MobileNetV3-Small")
print(f"Classes      : {num_classes}")

# ============================================================================
# TRAINING UTILITIES
# ============================================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

def validate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device))
            y_pred.append(out.argmax(1).item())
            y_true.append(y.item())
    return accuracy_score(y_true, y_pred)

# ============================================================================
# TRAINING LOOP
# ============================================================================
best_val_acc = 0.0
patience = 4
epochs_no_improve = 0

print("\n🚀 TRAINING STARTED")
for epoch in range(1, 8):
    model.train()
    epoch_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    val_acc = validate(model, val_loader)

    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {epoch_loss/len(train_loader):.4f} | "
          f"Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "checkpoints/best_model.pth")
        print("✅ Best model saved")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement ({epochs_no_improve}/{patience})")

    if epochs_no_improve >= patience:
        print("🛑 Early stopping")
        break

# ============================================================================
# FINAL TEST EVALUATION
# ============================================================================
print("\n" + "=" * 80)
print("🧪 FINAL TEST EVALUATION")
print("=" * 80)

model.load_state_dict(torch.load("checkpoints/best_model.pth"))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        y_pred.append(out.argmax(1).item())
        y_true.append(y.item())

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("\n📊 TEST METRICS")
print(f"Accuracy  : {acc*100:.2f}%")
print(f"Precision : {prec*100:.2f}%")
print(f"Recall    : {rec*100:.2f}%")
print(f"F1-score  : {f1*100:.2f}%")

print("\n✅ PIPELINE COMPLETE")


In [9]:
# ============================================================================
# FULL PIPELINE: TRAIN / VAL / TEST WITH EFFICIENTNET-LITE0 (MOBILE-FRIENDLY)
# ============================================================================

import os
import cv2
import re
import time
import torch
import shutil
import numpy as np
import torch.nn as nn
import torch.optim as optim

import timm
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================================================================
# MOBILE SIMULATION SETUP
# ============================================================================
device = "cpu"
torch.set_num_threads(4)
os.makedirs("checkpoints", exist_ok=True)

print("=" * 80)
print("📱 MOBILE SIMULATION MODE")
print(f"Device           : {device}")
print(f"CPU threads used : {torch.get_num_threads()}")
print("=" * 80)

# ============================================================================
# DATA TRANSFORMS
# ============================================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================================================
# IMAGE PROCESSING
# ============================================================================
def process_image(image_path, train=True):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    transform = train_transform if train else val_test_transform
    return transform(img)

# ============================================================================
# DATASET
# ============================================================================
class EarDataset(Dataset):
    """
    Dataset for images named:
    pig_[pig_id]_IMG_[img_id].jpg
    """

    def __init__(self, root_dir, train=True):
        self.root_dir = root_dir
        self.train = train
        self.images = []
        self.labels = []

        pattern = re.compile(r"pig_(\d+)_IMG_(\d+)", re.IGNORECASE)
        pig_ids = set()

        for fname in sorted(os.listdir(root_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            match = pattern.search(fname)
            if match is None:
                print(f"⚠️ Skipping malformed filename: {fname}")
                continue

            pig_id, _ = match.groups()
            self.images.append(os.path.join(root_dir, fname))
            self.labels.append(pig_id)
            pig_ids.add(pig_id)

        self.classes = sorted(list(pig_ids))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.labels = [self.class_to_idx[l] for l in self.labels]

        print(f"✅ Loaded images : {len(self.images)}")
        print(f"🏷️ Classes      : {len(self.classes)}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = process_image(self.images[idx], self.train)
        if img is None:
            img = torch.zeros(3, 224, 224)
        return img, self.labels[idx]

# ============================================================================
# PATHS
# ============================================================================
TRAIN_ROOT = "/kaggle/working/pig_dataset_split/training"
TEST_ROOT  = "/kaggle/working/pig_dataset_split/training"

print("\n📂 DATASET PATHS")
print(f"Train root : {TRAIN_ROOT}")
print(f"Test root  : {TEST_ROOT}")

# ============================================================================
# LOAD & SPLIT TRAIN DATA
# ============================================================================
full_train_dataset = EarDataset(TRAIN_ROOT, train=True)
num_classes = len(full_train_dataset.classes)

val_ratio = 0.15
val_size = int(len(full_train_dataset) * val_ratio)
train_size = len(full_train_dataset) - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset.dataset.train = True
val_dataset.dataset.train = False

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False)

# ============================================================================
# TEST DATA
# ============================================================================
test_dataset = EarDataset(TEST_ROOT, train=False)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

# ============================================================================
# MODEL (EfficientNet-Lite0)
# ============================================================================
model = timm.create_model(
    "efficientnet_lite0",
    pretrained=True,
    num_classes=num_classes
)

# Freeze backbone
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

model.to(device)

print("\n🧠 MODEL")
print("Architecture : EfficientNet-Lite0")
print(f"Classes      : {num_classes}")

# ============================================================================
# TRAINING UTILITIES
# ============================================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

def validate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device))
            y_pred.append(out.argmax(1).item())
            y_true.append(y.item())
    return accuracy_score(y_true, y_pred)

# ============================================================================
# TRAINING LOOP
# ============================================================================
best_val_acc = 0.0
patience = 4
epochs_no_improve = 0

print("\n🚀 TRAINING STARTED")
for epoch in range(1, 8):
    model.train()
    epoch_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    val_acc = validate(model, val_loader)

    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {epoch_loss/len(train_loader):.4f} | "
          f"Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "checkpoints/best_model.pth")
        print("✅ Best model saved")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement ({epochs_no_improve}/{patience})")

    if epochs_no_improve >= patience:
        print("🛑 Early stopping")
        break

# ============================================================================
# FINAL TEST EVALUATION
# ============================================================================
print("\n" + "=" * 80)
print("🧪 FINAL TEST EVALUATION")
print("=" * 80)

model.load_state_dict(torch.load("checkpoints/best_model.pth"))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        y_pred.append(out.argmax(1).item())
        y_true.append(y.item())

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("\n📊 TEST METRICS")
print(f"Accuracy  : {acc*100:.2f}%")
print(f"Precision : {prec*100:.2f}%")
print(f"Recall    : {rec*100:.2f}%")
print(f"F1-score  : {f1*100:.2f}%")

print("\n✅ PIPELINE COMPLETE")


📱 MOBILE SIMULATION MODE
Device           : cpu
CPU threads used : 4

📂 DATASET PATHS
Train root : /kaggle/working/pig_dataset_split/training
Test root  : /kaggle/working/pig_dataset_split/training
✅ Loaded images : 640
🏷️ Classes      : 20
✅ Loaded images : 640
🏷️ Classes      : 20


model.safetensors:   0%|          | 0.00/18.8M [00:00<?, ?B/s]


🧠 MODEL
Architecture : EfficientNet-Lite0
Classes      : 20

🚀 TRAINING STARTED
Epoch 01 | Train Loss: 2.9860 | Val Acc: 47.92%
✅ Best model saved
Epoch 02 | Train Loss: 1.4135 | Val Acc: 81.25%
✅ Best model saved
Epoch 03 | Train Loss: 0.7553 | Val Acc: 89.58%
✅ Best model saved
Epoch 04 | Train Loss: 0.4596 | Val Acc: 93.75%
✅ Best model saved
Epoch 05 | Train Loss: 0.2888 | Val Acc: 94.79%
✅ Best model saved
Epoch 06 | Train Loss: 0.2054 | Val Acc: 96.88%
✅ Best model saved
Epoch 07 | Train Loss: 0.1574 | Val Acc: 97.92%
✅ Best model saved

🧪 FINAL TEST EVALUATION

📊 TEST METRICS
Accuracy  : 99.22%
Precision : 99.23%
Recall    : 99.22%
F1-score  : 99.22%

✅ PIPELINE COMPLETE


In [13]:
def evaluate_loader(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device))
            y_pred.append(out.argmax(1).item())
            y_true.append(y.item())
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return acc, prec, rec, f1

In [36]:
aug_dirs = [
    d for d in glob.glob(os.path.join(TEST_ROOT, "*"))
    if os.path.isdir(d)
]

for aug_dir in aug_dirs:
    aug_dataset = EarDataset(
        aug_dir,
        train=False,
        organize_test_by_aug=False
    )
    aug_loader = DataLoader(
        aug_dataset,
        batch_size=1,
        shuffle=False
    )

    acc, prec, rec, f1 = evaluate_loader(model, aug_loader)

    aug_name = os.path.basename(aug_dir)

    print(f"\n📊 TEST METRICS - AUGMENTATION: {aug_name}")
    print(
        f"Accuracy  : {acc*100:.2f}% | "
        f"Precision : {prec*100:.2f}% | "
        f"Recall    : {rec*100:.2f}% | "
        f"F1-score  : {f1*100:.2f}%"
    )

    performance_dict["Accuracy"].append(round(acc * 100, 2))
    performance_dict["Precision"].append(round(prec * 100, 2))
    performance_dict["Recall"].append(round(rec * 100, 2))
    performance_dict["F1-score"].append(round(f1 * 100, 2))
    performance_dict["Augmentations"].append(aug_name)
    performance_dict["Models"].append("MobileViT-S")

print("\n✅ PIPELINE COMPLETE")


✅ Total images loaded     : 160
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['bright']

📊 TEST METRICS - AUGMENTATION: bright
Accuracy  : 98.12% | Precision : 98.33% | Recall    : 98.12% | F1-score  : 98.07%
✅ Total images loaded     : 160
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['dark']

📊 TEST METRICS - AUGMENTATION: dark
Accuracy  : 98.75% | Precision : 98.89% | Recall    : 98.75% | F1-score  : 98.70%
✅ Total images loaded     : 160
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['original']

📊 TEST METRICS - AUGMENTATION: original
Accuracy  : 99.38% | Precision : 99.44% | Recall    : 99.38% | F1-score  : 99.37%
✅ Total images loaded     : 160
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['blur']

📊 TEST METRICS - AUGMENTATION: blur
Accuracy  : 98.12% | Precision : 98.33% | Recall    : 98.12% | F1-score  : 98.07%
✅ Total images loaded     : 160
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['noise']

📊 TEST MET

In [37]:
performance_dict

{'Models': ['MobileViT-S',
  'MobileViT-S',
  'MobileViT-S',
  'MobileViT-S',
  'MobileViT-S',
  'MobileViT-S'],
 'Augmentations': ['Combined Perturbations',
  'bright',
  'dark',
  'original',
  'blur',
  'noise'],
 'Accuracy': [98.5, 98.12, 98.75, 99.38, 98.12, 98.12],
 'Precision': [98.58, 98.33, 98.89, 99.44, 98.33, 98.33],
 'Recall': [98.5, 98.12, 98.75, 99.38, 98.12, 98.12],
 'F1-score': [98.46, 98.07, 98.7, 99.37, 98.07, 98.12]}

In [ ]:
# ============================================================================
# FULL PIPELINE: TRAIN / VAL / TEST WITH MOBILE BENCHMARKING
# ============================================================================

import os
import cv2
import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================================================================
# MOBILE SIMULATION SETUP
# ============================================================================
device = "cpu"
torch.set_num_threads(4)
os.makedirs("checkpoints", exist_ok=True)

print("="*80)
print("📱 MOBILE SIMULATION MODE")
print(f"Device           : {device}")
print(f"CPU threads used : {torch.get_num_threads()}")
print("="*80)

# ============================================================================
# DATA TRANSFORMS
# ============================================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# ============================================================================
# IMAGE PROCESSING
# ============================================================================
def process_image_for_mobilenet(image_path, train=True):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    transform = train_transform if train else val_test_transform
    return transform(img)

# ============================================================================
# DATASET
# ============================================================================
import os
import re
import shutil
import torch
from torch.utils.data import Dataset


                
        
class EarDataset(Dataset):
    """
    Dataset for images named like:
    pig_[pig_id]_IMG_[img_id]_[augmentation].jpg

    - Label = pig_id
    - Optionally organizes test images into augmentation folders
    """
    def __init__(self, root_dir, train=True, organize_test_by_aug=True):
        self.root_dir = root_dir
        self.train = train
        self.images = []
        self.labels = []
        self.augmentations = []

        self.classes = []           # For compatibility
        self.class_to_idx = {}
        self.idx_to_class = {}

        # Regex pattern to capture pig_id, img_id, augmentation
        pattern = re.compile(r"pig_(\d+)_IMG_(\d+)_([a-zA-Z0-9]+)", re.IGNORECASE)

        pig_id_set = set()
        aug_set = set()

        # Optional: organize test images by augmentation
        if not train and organize_test_by_aug:
            print("🗂️  Organizing TEST images by augmentation type...")
            self._organize_by_augmentation(pattern)

        # Load images
        for fname in sorted(os.listdir(root_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            match = pattern.search(fname)
            if match is None:
                print(f"⚠️ Skipping malformed filename: {fname}")
                continue

            pig_id, img_id, aug = match.groups()

            self.images.append(os.path.join(root_dir, fname))
            self.labels.append(pig_id)
            self.augmentations.append(aug)
            pig_id_set.add(pig_id)
            aug_set.add(aug)

        # Build class mapping
        self.classes = sorted(list(pig_id_set))
        self.class_to_idx = {c:i for i, c in enumerate(self.classes)}
        self.idx_to_class = {i:c for c, i in self.class_to_idx.items()}

        # Convert labels to integers
        self.labels = [self.class_to_idx[lbl] for lbl in self.labels]

        # Logging
        print(f"✅ Total images loaded     : {len(self.images)}")
        print(f"🏷️  Unique pig_id classes  : {len(self.classes)}")
        print(f"🧪 Augmentation types     : {sorted(list(aug_set))}")
        print(f"📊 Example class mapping  :")
        for i, c in enumerate(self.classes[:5]):
            print(f"   {c} → class {self.class_to_idx[c]}")

    def _organize_by_augmentation(self, pattern):
        """Organize test images by augmentation"""
        for fname in os.listdir(self.root_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            match = pattern.search(fname)
            if match is None:
                continue

            # Correctly capture all three groups
            pig_id, img_id, aug = match.groups()

            aug_dir = os.path.join(self.root_dir, aug)
            os.makedirs(aug_dir, exist_ok=True)
            src = os.path.join(self.root_dir, fname)
            dst = os.path.join(aug_dir, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
        print("✅ Test images organized by augmentation")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = process_image_for_mobilenet(self.images[idx], self.train)
        if img is None:
            img = torch.zeros(3, 224, 224)
        label = self.labels[idx]
        return img, label

# ============================================================================
# PATHS
# ============================================================================
TRAIN_ROOT = "/kaggle/working/augmented_training"
TEST_ROOT  = "/kaggle/working/augmented_testing_combined"

print("\n📂 DATASET PATHS")
print(f"Train root : {TRAIN_ROOT}")
print(f"Test root  : {TEST_ROOT}")

# ============================================================================
# LOAD & SPLIT TRAIN DATA
# ============================================================================
full_train_dataset = EarDataset(TRAIN_ROOT, train=True)
num_classes = len(full_train_dataset.classes)

val_ratio = 0.15
total_size = len(full_train_dataset)
val_size = int(val_ratio * total_size)
train_size = total_size - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset.dataset.train = True
val_dataset.dataset.train = False

print("\n📊 DATA SPLIT")
print(f"Total train images : {total_size}")
print(f"Train set          : {train_size} (85%)")
print(f"Validation set     : {val_size} (15%)")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False)

# ============================================================================
# TEST DATA
# ============================================================================
test_dataset = EarDataset(TEST_ROOT, train=False)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

print(f"Test set           : {len(test_dataset)} images")


# MODEL
# ============================================================================
model = models.mobilenet_v3_small(pretrained=True)
model.classifier[3] = nn.Linear(1024, num_classes)

for p in model.features.parameters():
    p.requires_grad = False

model.to(device)

print("\n🧠 MODEL")
print("Architecture : MobileNetV3-Small")
print(f"Classes      : {num_classes}")


# TRAINING UTILITIES
# ============================================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

def validate_simple(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device))
            y_pred.append(out.argmax(1).item())
            y_true.append(y.item())
    return accuracy_score(y_true, y_pred)

# ============================================================================
# TRAINING LOOP
# ============================================================================
best_val_acc = 0.0
patience = 4
epochs_no_improve = 0

print("\n🚀 TRAINING STARTED")
for epoch in range(1, 8):
    model.train()
    epoch_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    val_acc = validate_simple(model, val_loader)

    print(f"Epoch {epoch:02d} | "
          f"Train Loss: {epoch_loss/len(train_loader):.4f} | "
          f"Val Acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "checkpoints/best_model.pth")
        print("✅ New best model saved")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement ({epochs_no_improve}/{patience})")

    if epochs_no_improve >= patience:
        print("🛑 Early stopping triggered")
        break

# ============================================================================
# FINAL TEST EVALUATION (HELD-OUT)
# ============================================================================
print("\n" + "="*80)
print("🧪 FINAL TEST SET EVALUATION (HELD-OUT)")
print("="*80)

model.load_state_dict(torch.load("checkpoints/best_model.pth"))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        y_pred.append(out.argmax(1).item())
        y_true.append(y.item())

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("\n📊 TEST METRICS")
print(f"Accuracy  : {acc*100:.2f}%")
print(f"Precision : {prec*100:.2f}%")
print(f"Recall    : {rec*100:.2f}%")
print(f"F1-score  : {f1*100:.2f}%")

performance_dict["Accuracy"].append(round(acc * 100, 2))
performance_dict["Precision"].append(round(prec * 100, 2))
performance_dict["Recall"].append(round(rec * 100, 2))
performance_dict["F1-score"].append(round(f1 * 100, 2))
performance_dict["Augmentations"].append(aug_name)
performance_dict["Models"].append("MobileNetV3-Small")

print("\n✅ PIPELINE COMPLETE")


📱 MOBILE SIMULATION MODE
Device           : cpu
CPU threads used : 4

📂 DATASET PATHS
Train root : /kaggle/working/augmented_training
Test root  : /kaggle/working/augmented_testing_combined
✅ Total images loaded     : 3200
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['blur', 'bright', 'dark', 'noise', 'original']
📊 Example class mapping  :
   05369 → class 0
   13865 → class 1
   13885 → class 2
   14005 → class 3
   14388 → class 4

📊 DATA SPLIT
Total train images : 3200
Train set          : 2720 (85%)
Validation set     : 480 (15%)
🗂️  Organizing TEST images by augmentation type...
✅ Test images organized by augmentation
✅ Total images loaded     : 800
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['blur', 'bright', 'dark', 'noise', 'original']
📊 Example class mapping  :
   05369 → class 0
   13865 → class 1
   13885 → class 2
   14005 → class 3
   14388 → class 4
Test set           : 800 images

🧠 MODEL
Architecture : MobileNetV3-Small
Classes      : 20



In [41]:
# ============================================================================
# TESTING PER AUGMENTATION TYPE
# ============================================================================
import glob

# Load the best trained model
model.load_state_dict(torch.load("checkpoints/best_model.pth"))
model.to(device)
model.eval()

# Get all augmentation folders inside the test root
aug_folders = [d for d in os.listdir(TEST_ROOT) 
               if os.path.isdir(os.path.join(TEST_ROOT, d))]

print("\n" + "="*80)
print("🧪 TESTING PER AUGMENTATION TYPE")
print("="*80)

for aug in sorted(aug_folders):
    aug_dir = os.path.join(TEST_ROOT, aug)
    print(f"\n📂 Augmentation: {aug}")
    
    # Create dataset and loader for this augmentation
    aug_dataset = EarDataset(aug_dir, train=False, organize_test_by_aug=False)
    aug_loader = DataLoader(aug_dataset, batch_size=1, shuffle=False)
    
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in aug_loader:
            out = model(x.to(device))
            y_pred.append(out.argmax(1).item())
            y_true.append(y.item())
    
    # Compute metrics
    if len(y_true) == 0:
        print("⚠️  No images found for this augmentation")
        continue
    
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    
    print(f"  Images   : {len(y_true)}")
    print(f"  Accuracy : {acc*100:.2f}%")
    print(f"  Precision: {prec*100:.2f}%")
    print(f"  Recall   : {rec*100:.2f}%")
    print(f"  F1-Score : {f1*100:.2f}%")

    performance_dict["Accuracy"].append(round(acc * 100, 2))
    performance_dict["Precision"].append(round(prec * 100, 2))
    performance_dict["Recall"].append(round(rec * 100, 2))
    performance_dict["F1-score"].append(round(f1 * 100, 2))
    performance_dict["Augmentations"].append(aug_name)
    performance_dict["Models"].append("MobileNetV3-Small")

print("\n✅ AUGMENTATION-WISE TESTING COMPLETE")



🧪 TESTING PER AUGMENTATION TYPE

📂 Augmentation: blur
✅ Total images loaded     : 160
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['blur']
📊 Example class mapping  :
   05369 → class 0
   13865 → class 1
   13885 → class 2
   14005 → class 3
   14388 → class 4
  Images   : 160
  Accuracy : 99.38%
  Precision: 99.44%
  Recall   : 99.38%
  F1-Score : 99.37%

📂 Augmentation: bright
✅ Total images loaded     : 160
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['bright']
📊 Example class mapping  :
   05369 → class 0
   13865 → class 1
   13885 → class 2
   14005 → class 3
   14388 → class 4
  Images   : 160
  Accuracy : 99.38%
  Precision: 99.44%
  Recall   : 99.38%
  F1-Score : 99.37%

📂 Augmentation: dark
✅ Total images loaded     : 160
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['dark']
📊 Example class mapping  :
   05369 → class 0
   13865 → class 1
   13885 → class 2
   14005 → class 3
   14388 → class 4
  Images   : 160
  Accuracy : 99.38%


In [15]:
# ============================================================================
# FULL PIPELINE: TRAIN / VAL / TEST WITH MOBILE BENCHMARKING
# ============================================================================

import os
import cv2
import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================================================================
# MOBILE SIMULATION SETUP
# ============================================================================
device = "cpu"
torch.set_num_threads(4)
os.makedirs("checkpoints", exist_ok=True)

print("="*80)
print("📱 MOBILE SIMULATION MODE")
print(f"Device           : {device}")
print(f"CPU threads used : {torch.get_num_threads()}")
print("="*80)

# ============================================================================
# DATA TRANSFORMS
# ============================================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

val_test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

# ============================================================================
# IMAGE PROCESSING
# ============================================================================
def process_image_for_mobilenet(image_path, train=True):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    transform = train_transform if train else val_test_transform
    return transform(img)

# ============================================================================
# DATASET
# ============================================================================
import os
import re
import shutil
import torch
from torch.utils.data import Dataset


                
        
class EarDataset(Dataset):
    """
    Dataset for images named like:
    pig_[pig_id]_IMG_[img_id]_[augmentation].jpg

    - Label = pig_id
    - Optionally organizes test images into augmentation folders
    """
    def __init__(self, root_dir, train=True, organize_test_by_aug=True):
        self.root_dir = root_dir
        self.train = train
        self.images = []
        self.labels = []
        self.augmentations = []

        self.classes = []           # For compatibility
        self.class_to_idx = {}
        self.idx_to_class = {}

        # Regex pattern to capture pig_id, img_id, augmentation
        pattern = re.compile(r"pig_(\d+)_IMG_(\d+)_([a-zA-Z0-9]+)", re.IGNORECASE)

        pig_id_set = set()
        aug_set = set()

        # Optional: organize test images by augmentation
        if not train and organize_test_by_aug:
            print("🗂️  Organizing TEST images by augmentation type...")
            self._organize_by_augmentation(pattern)

        # Load images
        for fname in sorted(os.listdir(root_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue

            match = pattern.search(fname)
            if match is None:
                print(f"⚠️ Skipping malformed filename: {fname}")
                continue

            pig_id, img_id, aug = match.groups()

            self.images.append(os.path.join(root_dir, fname))
            self.labels.append(pig_id)
            self.augmentations.append(aug)
            pig_id_set.add(pig_id)
            aug_set.add(aug)

        # Build class mapping
        self.classes = sorted(list(pig_id_set))
        self.class_to_idx = {c:i for i, c in enumerate(self.classes)}
        self.idx_to_class = {i:c for c, i in self.class_to_idx.items()}

        # Convert labels to integers
        self.labels = [self.class_to_idx[lbl] for lbl in self.labels]

        # Logging
        print(f"✅ Total images loaded     : {len(self.images)}")
        print(f"🏷️  Unique pig_id classes  : {len(self.classes)}")
        print(f"🧪 Augmentation types     : {sorted(list(aug_set))}")
        print(f"📊 Example class mapping  :")
        for i, c in enumerate(self.classes[:5]):
            print(f"   {c} → class {self.class_to_idx[c]}")

    def _organize_by_augmentation(self, pattern):
        """Organize test images by augmentation"""
        for fname in os.listdir(self.root_dir):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            match = pattern.search(fname)
            if match is None:
                continue

            # Correctly capture all three groups
            pig_id, img_id, aug = match.groups()

            aug_dir = os.path.join(self.root_dir, aug)
            os.makedirs(aug_dir, exist_ok=True)
            src = os.path.join(self.root_dir, fname)
            dst = os.path.join(aug_dir, fname)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
        print("✅ Test images organized by augmentation")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = process_image_for_mobilenet(self.images[idx], self.train)
        if img is None:
            img = torch.zeros(3, 224, 224)
        label = self.labels[idx]
        return img, label


def validate_simple(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device))
            y_pred.append(out.argmax(1).item())
            y_true.append(y.item())
    return accuracy_score(y_true, y_pred)



📱 MOBILE SIMULATION MODE
Device           : cpu
CPU threads used : 4


In [16]:
#### ============================================================================
# FULL PIPELINE: TRAIN / TEST WITH EFFICIENTNET-LITE0 ON ALREADY AUGMENTED DATA
# ============================================================================

import os, glob
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================================================================
# MOBILE SIMULATION SETUP
# ============================================================================
device = "cpu"
torch.set_num_threads(4)
os.makedirs("checkpoints", exist_ok=True)

print("="*80)
print("📱 MOBILE SIMULATION MODE")
print(f"Device           : {device}")
print(f"CPU threads used : {torch.get_num_threads()}")
print("="*80)

# ============================================================================
# PATHS
# ============================================================================
TRAIN_ROOT = "/kaggle/working/augmented_training"
TEST_ROOT  = "/kaggle/working/augmented_testing_combined"

# ============================================================================
# LOAD DATASETS
# ============================================================================
train_dataset = EarDataset(TRAIN_ROOT, train=True)
num_classes = len(train_dataset.classes)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Combined test dataset
test_dataset_combined = EarDataset(TEST_ROOT, train=False)
test_loader_combined = DataLoader(test_dataset_combined, batch_size=1, shuffle=False)

# ============================================================================
# MODEL SETUP
# ============================================================================
import timm
import torch.nn as nn

model = timm.create_model('efficientnet_lite0', pretrained=True, num_classes=num_classes)

# Freeze all feature layers (optional)
for name, param in model.named_parameters():
    if "classifier" not in name:  # keep classifier trainable
        param.requires_grad = False

model.to(device)

print(f"🧠 MODEL: EfficientNet-Lite0 | Classes: {num_classes}")

# ============================================================================
# TRAINING SETUP
# ============================================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

# ============================================================================
# HELPER FUNCTION TO EVALUATE A LOADER
# ============================================================================
def evaluate_loader(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x.to(device))
            y_pred.append(out.argmax(1).item())
            y_true.append(y.item())
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return acc, prec, rec, f1

# ============================================================================
# TRAINING LOOP
# ============================================================================
best_loss = float('inf')
for epoch in range(1, 8):
    model.train()
    epoch_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch:02d} | Train Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), "checkpoints/best_model.pth")
        print("✅ Model checkpoint saved")

# ============================================================================
# FINAL TEST EVALUATION
# ============================================================================
model.load_state_dict(torch.load("checkpoints/best_model.pth"))
model.eval()

# Combined augmentations
acc, prec, rec, f1 = evaluate_loader(model, test_loader_combined)
print("\n📊 TEST METRICS - COMBINED AUGMENTATIONS")
print(f"Accuracy  : {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}%")
performance_dict["Accuracy"].append(round(acc * 100, 2))
performance_dict["Precision"].append(round(prec * 100, 2))
performance_dict["Recall"].append(round(rec * 100, 2))
performance_dict["F1-score"].append(round(f1 * 100, 2))
performance_dict["Augmentations"].append(aug_name)
performance_dict["Models"].append("EfficientNet-lite0")

# Evaluate each augmentation separately
aug_dirs = [d for d in glob.glob(os.path.join(TEST_ROOT, "*")) if os.path.isdir(d)]
for aug_dir in aug_dirs:
    aug_dataset = EarDataset(aug_dir, train=False, organize_test_by_aug=False)
    aug_loader = DataLoader(aug_dataset, batch_size=1, shuffle=False)
    acc, prec, rec, f1 = evaluate_loader(model, aug_loader)
    print(f"\n📊 TEST METRICS - AUGMENTATION: {os.path.basename(aug_dir)}")
    print(f"Accuracy  : {acc*100:.2f}% | Precision: {prec*100:.2f}% | Recall: {rec*100:.2f}% | F1: {f1*100:.2f}%")

    performance_dict["Accuracy"].append(round(acc * 100, 2))
    performance_dict["Precision"].append(round(prec * 100, 2))
    performance_dict["Recall"].append(round(rec * 100, 2))
    performance_dict["F1-score"].append(round(f1 * 100, 2))
    performance_dict["Augmentations"].append(aug_name)
    performance_dict["Models"].append("EfficientNet-lite0")

print("\n✅ PIPELINE COMPLETE")


📱 MOBILE SIMULATION MODE
Device           : cpu
CPU threads used : 4
✅ Total images loaded     : 3200
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['blur', 'bright', 'dark', 'noise', 'original']
📊 Example class mapping  :
   05369 → class 0
   13865 → class 1
   13885 → class 2
   14005 → class 3
   14388 → class 4
🗂️  Organizing TEST images by augmentation type...
✅ Test images organized by augmentation
✅ Total images loaded     : 800
🏷️  Unique pig_id classes  : 20
🧪 Augmentation types     : ['blur', 'bright', 'dark', 'noise', 'original']
📊 Example class mapping  :
   05369 → class 0
   13865 → class 1
   13885 → class 2
   14005 → class 3
   14388 → class 4


model.safetensors:   0%|          | 0.00/18.8M [00:00<?, ?B/s]

🧠 MODEL: EfficientNet-Lite0 | Classes: 20
Epoch 01 | Train Loss: 1.6007
✅ Model checkpoint saved
Epoch 02 | Train Loss: 0.3347
✅ Model checkpoint saved
Epoch 03 | Train Loss: 0.1627
✅ Model checkpoint saved
Epoch 04 | Train Loss: 0.0995
✅ Model checkpoint saved
Epoch 05 | Train Loss: 0.0704
✅ Model checkpoint saved
Epoch 06 | Train Loss: 0.0524
✅ Model checkpoint saved
Epoch 07 | Train Loss: 0.0407
✅ Model checkpoint saved

📊 TEST METRICS - COMBINED AUGMENTATIONS
Accuracy  : 99.75% | Precision: 99.76% | Recall: 99.75% | F1: 99.75%


NameError: name 'performance_dict' is not defined

In [ ]:
def extract_ear_region(img_rgb):
    """
    Extract the inner ear region using red channel dominance.
    
    Args:
        img_rgb (numpy.ndarray): RGB image
        
    Returns:
        tuple: (masked_inner_ear, filled_mask)
    """
    # Extract red channel dominance for inner ear area
    r_dominance = np.zeros_like(img_rgb[:,:,0])
    r_dominance[(img_rgb[:,:,0] > 60) & 
                (img_rgb[:,:,0] > img_rgb[:,:,1]*1.7) & 
                (img_rgb[:,:,0] > img_rgb[:,:,2]*1.7)] = 255
    
    # Clean up the mask
    kernel = np.ones((5, 5), np.uint8)
    r_dominance_cleaned = cv2.morphologyEx(r_dominance, cv2.MORPH_CLOSE, kernel)
    r_dominance_cleaned = cv2.morphologyEx(r_dominance_cleaned, cv2.MORPH_OPEN, kernel)
    
    # Find connected components to isolate the main inner part
    labeled_array, num_features = ndimage.label(r_dominance_cleaned)
    component_sizes = np.bincount(labeled_array.ravel())
    component_sizes[0] = 0  # Ignore background
    
    # Check if any components were found
    if len(component_sizes) <= 1:
        print("No ear region detected.")
        return None, None
    
    largest_component_label = np.argmax(component_sizes)
    inner_ear_mask = (labeled_array == largest_component_label).astype(np.uint8) * 255
    
    # Fill any holes in the mask
    contours, _ = cv2.findContours(inner_ear_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled_mask = np.zeros_like(inner_ear_mask)
    for contour in contours:
        cv2.drawContours(filled_mask, [contour], -1, 255, thickness=cv2.FILLED)
    
    # Apply the mask to get the inner ear region
    masked_inner_ear = cv2.bitwise_and(img_rgb, img_rgb, mask=filled_mask)
    
    return masked_inner_ear, filled_mask

print("Ear region extraction function defined!")

In [ ]:
# Load a sample image
sample_image_path = os.path.join(train_dir, os.listdir(train_dir)[0])
img = cv2.imread(sample_image_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Extract ear region
masked_inner_ear, filled_mask = extract_ear_region(img_rgb)

# Display results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(img_rgb)
plt.title('Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(filled_mask, cmap='gray')
plt.title('Inner Ear Mask')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(masked_inner_ear)
plt.title('Masked Inner Ear')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def extract_vein_features(masked_inner_ear, filled_mask):
    """
    Extract vein features from the masked inner ear.
    
    Args:
        masked_inner_ear (numpy.ndarray): Masked inner ear RGB image
        filled_mask (numpy.ndarray): Binary mask of the inner ear region
        
    Returns:
        tuple: (final_veins, vein_features)
    """
    # Step 1: Extract individual channels
    r, g, b = cv2.split(masked_inner_ear)
    
    # Step 2: Invert red channel since veins appear darker
    r_inv = 255 - r
    
    # Step 3: Apply contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    r_enhanced = clahe.apply(r_inv)
    
    # Step 4: Apply image sharpening
    kernel_sharpen = np.array([[0, -1,  0],
                   [-1,  5, -1],
                   [0, -1,  0]])
    r_sharpened = cv2.filter2D(r_enhanced, -1, kernel_sharpen)
    
    # Step 5: Use adaptive threshold
    r_adaptive = cv2.adaptiveThreshold(
        r_sharpened, 255, cv2.ADAPTIVE_THRESH_MEAN_C, 
        cv2.THRESH_BINARY, 51, 0
    )
    
    # Step 6: Initial morphological cleaning
    vein_kernel = np.ones((3, 3), np.uint8)
    opened = cv2.morphologyEx(r_adaptive, cv2.MORPH_OPEN, vein_kernel)
    
    # Step 7: Apply stronger small object removal
    veins_binary = opened > 0
    veins_filtered_initial = remove_small_objects(veins_binary, min_size=300)
    
    # Step 8: Dilate to connect closely related segments
    dilated = cv2.dilate(veins_filtered_initial.astype(np.uint8) * 255, vein_kernel, iterations=1)
    
    # Step 9: Use connected component analysis
    labeled, num_components = ndimage.label(dilated)
    component_sizes = np.bincount(labeled.ravel())
    component_sizes[0] = 0  # Ignore background
    
    # Step 10: Keep only the largest vein structures
    sorted_indices = np.argsort(component_sizes)[::-1]  # Sort descending
    N = 15  # Number of major veins to keep
    major_veins = np.zeros_like(dilated)
    for i in range(min(N, len(sorted_indices))):
        idx = sorted_indices[i]
        if component_sizes[idx] > 30:  # Minimum size for a significant vein
            major_veins[labeled == idx] = 255
    
    # Step 11: Final cleaning and edge detection
    edges = cv2.Canny(major_veins, 30, 100)
    dilated_edges = cv2.dilate(edges, np.ones((2, 2), np.uint8), iterations=1)
    veins_with_edges = cv2.bitwise_or(major_veins, dilated_edges)
    
    # Step 12: Final mask application and size-based filtering
    veins_binary_final = veins_with_edges > 0
    veins_filtered_final = remove_small_objects(veins_binary_final, min_size=300)
    veins_filtered_uint8 = (veins_filtered_final * 255).astype(np.uint8)
    
    # Apply inner ear mask for the final result
    final_veins = cv2.bitwise_and(veins_filtered_uint8, veins_filtered_uint8, mask=filled_mask)
    
    # Convert to skeleton for feature extraction
    skeleton = skeletonize(final_veins > 0)
    skeleton_uint8 = (skeleton * 255).astype(np.uint8)
    
    # Extract vein features (bifurcation points and endpoints)
    points = np.column_stack(np.where(skeleton > 0))
    
    if len(points) == 0:
        print("No skeleton points found.")
        return final_veins, None
    
    # For each point, count its neighbors to identify bifurcations and endpoints
    bifurcation_points = []
    endpoints = []
    
    for y, x in points:
        if y > 0 and y < skeleton.shape[0]-1 and x > 0 and x < skeleton.shape[1]-1:
            neighborhood = skeleton[y-1:y+2, x-1:x+2].copy()
            neighborhood[1, 1] = 0  # Exclude center
            neighbor_count = np.count_nonzero(neighborhood)
            
            if neighbor_count > 2:
                bifurcation_points.append((x, y))  # Bifurcation point
            elif neighbor_count == 1:
                endpoints.append((x, y))  # Endpoint
    
    # Sample points along the skeleton
    num_points = 100
    if len(points) > num_points:
        indices = np.linspace(0, len(points) - 1, num_points, dtype=int)
        sampled_points = points[indices]
        sampled_points = [(pt[1], pt[0]) for pt in sampled_points]  # Swap x and y
    else:
        sampled_points = [(pt[1], pt[0]) for pt in points]  # Swap x and y
    
    vein_features = {
        "bifurcation_points": bifurcation_points,
        "endpoints": endpoints,
        "sampled_points": sampled_points
    }
    
    return final_veins, vein_features, skeleton_uint8

print("Vein extraction function defined!")

In [ ]:
# Extract veins from the sample image
final_veins, vein_features, skeleton = extract_vein_features(masked_inner_ear, filled_mask)

# Display vein extraction process
plt.figure(figsize=(20, 10))

# Original and masked ear
plt.subplot(2, 4, 1)
plt.imshow(img_rgb)
plt.title('Original Image')
plt.axis('off')

plt.subplot(2, 4, 2)
plt.imshow(masked_inner_ear)
plt.title('Masked Inner Ear')
plt.axis('off')

# Get processing steps for visualization
r, g, b = cv2.split(masked_inner_ear)
r_inv = 255 - r
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
r_enhanced = clahe.apply(r_inv)
kernel_sharpen = np.array([[-1,-1,-1], [-1, 9,-1], [-1,-1,-1]])
r_sharpened = cv2.filter2D(r_enhanced, -1, kernel_sharpen)
r_adaptive = cv2.adaptiveThreshold(r_sharpened, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 51, 0)

# Display processing steps
plt.subplot(2, 4, 3)
plt.imshow(r_enhanced, cmap='gray')
plt.title('Enhanced Red Channel')
plt.axis('off')

plt.subplot(2, 4, 4)
plt.imshow(r_adaptive, cmap='gray')
plt.title('Adaptive Threshold')
plt.axis('off')

plt.subplot(2, 4, 5)
plt.imshow(final_veins, cmap='gray')
plt.title('Filtered Veins')
plt.axis('off')

plt.subplot(2, 4, 6)
plt.imshow(skeleton, cmap='gray')
plt.title('Vein Skeleton')
plt.axis('off')

# Display feature visualization
plt.subplot(2, 4, 7)
plt.imshow(skeleton, cmap='gray')
plt.title('Vein Features')

# Add bifurcation points and endpoints
if vein_features:
    bifurcation_points = vein_features["bifurcation_points"]
    endpoints = vein_features["endpoints"]
    
    if bifurcation_points:
        bifurcation_points = np.array(bifurcation_points)
        plt.scatter(bifurcation_points[:, 0], bifurcation_points[:, 1], 
                   color='red', s=15, label='Bifurcation Points')
    
    if endpoints:
        endpoints = np.array(endpoints)
        plt.scatter(endpoints[:, 0], endpoints[:, 1], 
                   color='blue', s=15, label='Endpoints')
    
    plt.legend(loc='upper right', fontsize=8)

plt.axis('off')

# Overlay on original for verification
plt.subplot(2, 4, 8)
overlay = masked_inner_ear.copy()
if skeleton is not None:
    overlay[skeleton > 0] = [255, 0, 0]  # Red for veins
plt.imshow(overlay)
plt.title('Veins Overlay')
plt.axis('off')

plt.tight_layout()
plt.show()

# Print feature statistics
if vein_features:
    print(f"Extracted Features:")
    print(f"- Bifurcation Points: {len(vein_features['bifurcation_points'])}")
    print(f"- Endpoints: {len(vein_features['endpoints'])}")
    print(f"- Sampled Points: {len(vein_features['sampled_points'])}")

In [ ]:
def generate_feature_vector(features):
    """
    Generate a normalized feature vector from extracted features.
    
    Args:
        features (dict): Dictionary containing feature points
        
    Returns:
        numpy.ndarray: Feature vector
    """
    if features is None:
        return np.zeros(68)  # Return zeros if no features
    
    # Convert points to arrays for easier manipulation
    bifurcation_points = np.array(features["bifurcation_points"]) if len(features["bifurcation_points"]) > 0 else np.array([[0, 0]])
    endpoints = np.array(features["endpoints"]) if len(features["endpoints"]) > 0 else np.array([[0, 0]])
    sampled_points = np.array(features["sampled_points"])
    
    # Calculate statistics from bifurcation points
    if len(bifurcation_points) > 1:
        # Distance between bifurcation points
        bifurcation_distances = cdist(bifurcation_points, bifurcation_points)
        mean_bifurcation_distance = np.mean(bifurcation_distances[bifurcation_distances > 0]) if np.sum(bifurcation_distances > 0) > 0 else 0
        std_bifurcation_distance = np.std(bifurcation_distances[bifurcation_distances > 0]) if np.sum(bifurcation_distances > 0) > 0 else 0
        
        # Angle statistics
        angles = []
        for i in range(len(bifurcation_points)):
            for j in range(i+1, len(bifurcation_points)):
                if i != j:
                    angle = np.arctan2(bifurcation_points[j, 1] - bifurcation_points[i, 1],
                                       bifurcation_points[j, 0] - bifurcation_points[i, 0])
                    angles.append(angle)
        
        angle_hist = np.histogram(angles, bins=8, range=(-np.pi, np.pi))[0]
        angle_hist = angle_hist / (np.sum(angle_hist) + 1e-10)  # Normalize
    else:
        mean_bifurcation_distance = 0
        std_bifurcation_distance = 0
        angle_hist = np.zeros(8)
    
    # Calculate statistics from endpoints
    if len(endpoints) > 1:
        endpoint_distances = cdist(endpoints, endpoints)
        mean_endpoint_distance = np.mean(endpoint_distances[endpoint_distances > 0]) if np.sum(endpoint_distances > 0) > 0 else 0
        std_endpoint_distance = np.std(endpoint_distances[endpoint_distances > 0]) if np.sum(endpoint_distances > 0) > 0 else 0
    else:
        mean_endpoint_distance = 0
        std_endpoint_distance = 0
    
    # Calculate spatial distribution statistics
    if len(sampled_points) > 0:
        x_coords = sampled_points[:, 0]
        y_coords = sampled_points[:, 1]
        
        x_mean, x_std = np.mean(x_coords), np.std(x_coords)
        y_mean, y_std = np.mean(y_coords), np.std(y_coords)
        
        # Create spatial density histogram
        x_bins = 10
        y_bins = 10
        hist_2d = np.histogram2d(x_coords, y_coords, bins=[x_bins, y_bins])[0]
        hist_2d = hist_2d.flatten() / (np.sum(hist_2d) + 1e-10)  # Normalize
    else:
        x_mean, x_std = 0, 0
        y_mean, y_std = 0, 0
        hist_2d = np.zeros(100)
    
    # Combine all features into a single vector
    feature_vector = np.concatenate([
        [len(bifurcation_points), len(endpoints)],  # Counts
        [mean_bifurcation_distance, std_bifurcation_distance],  # Bifurcation statistics
        [mean_endpoint_distance, std_endpoint_distance],  # Endpoint statistics
        [x_mean, x_std, y_mean, y_std],  # Spatial distribution
        angle_hist,  # Angle histogram
        hist_2d[:50]  # Spatial density (truncated to 50 elements)
    ])
    
    return feature_vector

print("Feature vector generation function defined!")

In [ ]:
# Generate feature vector for the sample image
if vein_features:
    feature_vector = generate_feature_vector(vein_features)
    
    # Display the feature vector
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(feature_vector)
    plt.title('Feature Vector')
    plt.xlabel('Feature Index')
    plt.ylabel('Value')
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.bar(range(len(feature_vector)), feature_vector)
    plt.title('Feature Vector (Bar Plot)')
    plt.xlabel('Feature Index')
    plt.ylabel('Value')
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Feature vector shape: {feature_vector.shape}")
    print(f"Feature vector statistics:")
    print(f"- Min: {feature_vector.min():.4f}")
    print(f"- Max: {feature_vector.max():.4f}")
    print(f"- Mean: {feature_vector.mean():.4f}")
    print(f"- Std: {feature_vector.std():.4f}")

In [ ]:
def process_image(image_path, output_dir=None, save_features=False, display=False):
    """
    Complete processing pipeline for a single image.
    
    Args:
        image_path (str): Path to the image
        output_dir (str, optional): Directory to save processed results
        save_features (bool): Whether to save extracted features
        display (bool): Whether to display visualizations
        
    Returns:
        tuple: (feature_vector, features_dict)
    """
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Could not read image: {image_path}")
        return None, None
    
    # Convert to RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Extract ear region
    masked_inner_ear, filled_mask = extract_ear_region(img_rgb)
    if masked_inner_ear is None:
        print(f"Failed to extract ear region from {image_path}")
        return None, None
    
    # Extract vein features
    final_veins, vein_features, skeleton = extract_vein_features(masked_inner_ear, filled_mask)
    if vein_features is None:
        print(f"Failed to extract vein features from {image_path}")
        return None, None
    
    # Generate feature vector
    feature_vector = generate_feature_vector(vein_features)
    
    # Save if required
    if output_dir and save_features:
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        
        # Save images
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        
        # Save masked ear
        cv2.imwrite(os.path.join(output_dir, f"{base_name}_masked.jpg"), 
                    cv2.cvtColor(masked_inner_ear, cv2.COLOR_RGB2BGR))
        
        # Save vein image
        cv2.imwrite(os.path.join(output_dir, f"{base_name}_veins.png"), final_veins)
        
        # Save features
        with open(os.path.join(output_dir, f"{base_name}_features.pkl"), 'wb') as f:
            pickle.dump(vein_features, f)
        
        # Save feature vector
        np.save(os.path.join(output_dir, f"{base_name}_vector.npy"), feature_vector)
    
    # Display if required
    if display:
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        plt.imshow(img_rgb)
        plt.title('Original')
        plt.axis('off')
        
        plt.subplot(1, 3, 2)
        plt.imshow(masked_inner_ear)
        plt.title('Masked Inner Ear')
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        plt.imshow(final_veins, cmap='gray')
        plt.title('Extracted Veins')
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
    
    return feature_vector, vein_features

print("Complete image processing function defined!")

In [ ]:
# Process a sample image
processed_dir = "/kaggle/working/processed_samples"
sample_feature_vector, sample_features = process_image(
    sample_image_path, 
    output_dir=processed_dir,
    save_features=True,
    display=True
)

print(f"Sample image processed and features saved to {processed_dir}")

In [ ]:
def build_database(train_dir, output_dir=None, display_progress=True):
    """
    Build a database of feature vectors from training images.
    
    Args:
        train_dir (str): Directory containing training images
        output_dir (str, optional): Directory to save processed results
        display_progress (bool): Whether to display progress
        
    Returns:
        dict: Database mapping pig IDs to feature vectors
    """
    # Dictionary to store feature vectors for each pig
    database = {}
    
    # Get all image paths
    image_paths = [str(p) for p in Path(train_dir).glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]

    print(f"Found {len(image_paths)} images in training directory")
    
    for i, img_path in enumerate(image_paths):
        # Extract pig ID from filename (assuming format pig_ID_IMAGENUMBER.jpg)
        filename = os.path.basename(img_path)
        parts = filename.split('_')
        
        if len(parts) >= 2:
            # Extract pig ID (assumes pig_<ID>_*.jpg format)
            pig_id = parts[1]
            
            if display_progress and i % 20 == 0:
                print(f"Processing image {i+1}/{len(image_paths)} for pig {pig_id}: {filename}")

            
            try:
                # Process image and get feature vector
                feature_vector, _ = process_image(img_path, output_dir, save_features=True, display=False)
                
                if feature_vector is not None:
                    # Store in database
                    if pig_id not in database:
                        database[pig_id] = []
                    
                    database[pig_id].append(feature_vector)
                
            except Exception as e:
                print(f"Error processing image {img_path}: {e}")
    
    # Calculate average feature vector for each pig
    for pig_id in database:
        if len(database[pig_id]) > 0:
            database[pig_id] = np.mean(database[pig_id], axis=0)
    
    # Save database if output directory is provided
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        database_path = os.path.join(output_dir, "pig_database.pkl")
        with open(database_path, 'wb') as f:
            pickle.dump(database, f)
        
        print(f"Database saved with {len(database)} pigs to {database_path}")
    
    return database

print("Database building function defined!")

In [ ]:
def match_image(test_image_path, database, top_n=3, display=False):
    """
    Match a test image against the database and return the top N matches.
    
    Args:
        test_image_path (str): Path to the test image
        database (dict): Database of feature vectors by pig ID
        top_n (int): Number of top matches to return
        display (bool): Whether to display visualizations
        
    Returns:
        list: List of (pig_id, distance) tuples sorted by distance
    """
    # Process test image
    test_vector, _ = process_image(test_image_path, display=display)
    
    if test_vector is None:
        print(f"Failed to process test image {test_image_path}")
        return []
    
    # Calculate distances to all pigs in database
    distances = []
    for pig_id, reference_vector in database.items():
        # Use Euclidean distance
        distance = np.linalg.norm(test_vector - reference_vector)
        distances.append((pig_id, distance))
    
    # Sort by distance (smaller is better)
    distances.sort(key=lambda x: x[1])
    
    # Return top N matches
    return distances[:top_n]

print("Matching function defined!")

In [ ]:
from sklearn import svm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import joblib

def build_svm_database(train_dir, output_dir=None, display_progress=True):
    """
    Build a database using SVM classifier from training images.
    
    Args:
        train_dir (str): Directory containing training images
        output_dir (str, optional): Directory to save processed results
        display_progress (bool): Whether to display progress
        
    Returns:
        tuple: (svm_model, scaler, pig_ids)
    """
    feature_vectors = []
    pig_ids = []
    image_paths = [str(p) for p in Path(train_dir).glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]

    for i, img_path in enumerate(image_paths):
        filename = os.path.basename(img_path)
        parts = filename.split('_')
        if len(parts) >= 2:
            pig_id = parts[1]
            feature_vector, _ = process_image(img_path, output_dir, save_features=False, display=False)
            if feature_vector is not None:
                feature_vectors.append(feature_vector)
                pig_ids.append(pig_id)
            if display_progress and i % 40 == 0:
                print(f"Processed {i+1}/{len(image_paths)} images")

    # Standardize features
    scaler = StandardScaler()
    X = scaler.fit_transform(feature_vectors)
    y = pig_ids

    # Train SVM
    clf = svm.SVC(kernel='rbf', probability=True)
    clf.fit(X, y)

    # Save model and scaler
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        joblib.dump(clf, os.path.join(output_dir, "svm_model.pkl"))
        joblib.dump(scaler, os.path.join(output_dir, "svm_scaler.pkl"))
        print(f"SVM model and scaler saved to {output_dir}")

    return clf, scaler, pig_ids

print("SVM database building function defined!")

In [ ]:
def verify_accuracy(test_dir, database, display_progress=True):
    """
    Calculate verification accuracy on test set.
    
    Args:
        test_dir (str): Directory containing test images
        database (dict): Database of feature vectors by pig ID
        display_progress (bool): Whether to display progress
        
    Returns:
        float: Accuracy as a percentage
    """
    # Get all test images
    all_files = glob.glob(os.path.join(test_dir, "*"))
    test_images = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Found {len(test_images)} images in test directory")
    
    correct = 0
    total = 0
    
    # Dictionary to store results by pig
    results_by_pig = {}
    
    for i, img_path in enumerate(test_images):
        # Extract actual pig ID from filename
        filename = os.path.basename(img_path)
        parts = filename.split('_')
        
        if len(parts) >= 2:
            actual_pig_id = parts[1]
            
            if display_progress and i % 10 == 0:
                print(f"Testing image {i+1}/{len(test_images)}: {filename}")
            
            try:
                # Match against database
                matches = match_image(img_path, database, top_n=1, display=False)
                
                if matches:
                    predicted_pig_id = matches[0][0]
                    distance = matches[0][1]
                    
                    # Update statistics
                    is_correct = actual_pig_id == predicted_pig_id
                    
                    if is_correct:
                        correct += 1
                    
                    total += 1
                    
                    # Update results by pig
                    if actual_pig_id not in results_by_pig:
                        results_by_pig[actual_pig_id] = {
                            'correct': 0,
                            'total': 0,
                            'distances': []
                        }
                    
                    results_by_pig[actual_pig_id]['total'] += 1
                    if is_correct:
                        results_by_pig[actual_pig_id]['correct'] += 1
                    results_by_pig[actual_pig_id]['distances'].append(distance)
                    
                    if display_progress:
                        print(f"  Prediction: {predicted_pig_id}, Actual: {actual_pig_id}, Correct: {is_correct}")
                
            except Exception as e:
                print(f"Error processing test image {img_path}: {e}")
    
    # Calculate overall accuracy
    accuracy = (correct / total * 100) if total > 0 else 0
    
    # Print overall results
    print("\nVerification Results:")
    print(f"Overall Accuracy: {accuracy:.2f}% ({correct}/{total})")
    
    # Print results by pig
    print("\nResults by pig:")
    for pig_id, results in results_by_pig.items():
        pig_accuracy = (results['correct'] / results['total'] * 100) if results['total'] > 0 else 0
        avg_distance = np.mean(results['distances']) if results['distances'] else 0
        print(f"Pig {pig_id}: {pig_accuracy:.2f}% ({results['correct']}/{results['total']}), Avg Distance: {avg_distance:.4f}")
    
    return accuracy

print("Accuracy verification function defined!")

In [ ]:
# Build SVM database using training images
import pandas as pd
svm_database_dir = "/kaggle/working/svm_database_new"
svm_model, svm_scaler, svm_pig_ids = build_svm_database(train_dir, output_dir=svm_database_dir)

def verify_svm_accuracy(test_dir, svm_model, svm_scaler, pig_ids, display_progress=True):
    """
    Calculate verification accuracy using SVM model on test set.
    
    Args:
        test_dir (str): Directory containing test images
        svm_model: Trained SVM model
        svm_scaler: Feature scaler
        pig_ids: List of pig IDs corresponding to training labels
        display_progress (bool): Whether to display progress
        
    Returns:
        float: Accuracy as a percentage
    """
    all_files = glob.glob(os.path.join(test_dir, "*"))
    test_images = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Found {len(test_images)} images in test directory")
    
    correct = 0
    total = 0
    performance_dict = {}
    
    for i, img_path in enumerate(test_images):
        filename = os.path.basename(img_path)
        parts = filename.split('_')
        actual_pig_id = parts[1] if len(parts) >= 2 else "Unknown"
        
        feature_vector, _ = process_image(img_path, display=False)
        if feature_vector is not None:
            X_test = svm_scaler.transform([feature_vector])
            pred_pig_id = svm_model.predict(X_test)[0]
            is_correct = pred_pig_id == actual_pig_id
            if is_correct:
                correct += 1
            total += 1
            print(f" Testing the model {i+10} /{total}")
            if 'actual' not in performance_dict:
                performance_dict['actual'] = [actual_pig_id]
            else:
                performance_dict['actual'].append(actual_pig_id)
            if 'predicted' not in performance_dict:
                performance_dict['predicted'] = [pred_pig_id]
            else:
                performance_dict['predicted'].append(pred_pig_id)
            if 'correct' not in performance_dict:
                performance_dict['correct'] = [is_correct]
            else:
                performance_dict['correct'].append(is_correct)
    performance_df= pd.DataFrame(performance_dict)
    
    accuracy = (correct / total * 100) if total > 0 else 0
    print(f"\nSVM Verification Accuracy: {accuracy:.2f}% ({correct}/{total})")
    return performance_df, accuracy

# Run SVM accuracy verification
performance_df, svm_accuracy = verify_svm_accuracy(test_dir, svm_model, svm_scaler, svm_pig_ids)
performance_df

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
y_true = performance_df["actual"]
y_pred = performance_df["predicted"]

precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall_macro    = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1_macro        = f1_score(y_true, y_pred, average="macro", zero_division=0)

precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall_weighted    = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1_weighted        = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"Macro Precision: {precision_macro:.4f}")
print(f"Macro Recall: {recall_macro:.4f}")
print(f"Macro F1: {f1_macro:.4f}")

print(f"\nWeighted Precision: {precision_weighted:.4f}")
print(f"Weighted Recall: {recall_weighted:.4f}")
print(f"Weighted F1: {f1_weighted:.4f}")


In [ ]:
# import glob
# import os
# import pandas as pd

# def verify_svm_accuracy(test_dir, svm_model, svm_scaler, pig_ids,
#                         suffix_filter=None, display_progress=True):
#     """
#     Calculate verification accuracy using SVM model on a specific test scenario.
    
#     Args:
#         test_dir (str): Directory containing test images
#         suffix_filter (str): One of ['original','noise','blur','dark','bright']
        
#     Returns:
#         DataFrame, accuracy (%)
#     """
    
#     all_files = glob.glob(os.path.join(test_dir, "*"))
    
#     # Filter images by suffix
#     if suffix_filter is not None:
#         test_images = [
#             f for f in all_files
#             if f.lower().endswith(('.jpg', '.jpeg', '.png'))
#             and f"_{suffix_filter}" in os.path.basename(f)
#         ]
#     else:
#         test_images = [
#             f for f in all_files
#             if f.lower().endswith(('.jpg', '.jpeg', '.png'))
#         ]
    
#     print(f"\n▶ Testing scenario: {suffix_filter}")
#     print(f"Found {len(test_images)} images")

#     correct = 0
#     total = 0
#     performance_dict = {"actual": [], "predicted": [], "correct": []}

#     for i, img_path in enumerate(test_images):
#         filename = os.path.basename(img_path)
#         parts = filename.split('_')
#         actual_pig_id = parts[1] if len(parts) >= 2 else "Unknown"

#         feature_vector, _ = process_image(img_path, display=False)
#         if feature_vector is not None:
#             X_test = svm_scaler.transform([feature_vector])
#             pred_pig_id = svm_model.predict(X_test)[0]
#             is_correct = pred_pig_id == actual_pig_id

#             correct += int(is_correct)
#             total += 1

#             performance_dict["actual"].append(actual_pig_id)
#             performance_dict["predicted"].append(pred_pig_id)
#             performance_dict["correct"].append(is_correct)

#     performance_df = pd.DataFrame(performance_dict)
#     accuracy = (correct / total * 100) if total > 0 else 0

#     print(f"SVM Accuracy ({suffix_filter}): {accuracy:.2f}% ({correct}/{total})")
#     return performance_df, accuracy


In [ ]:
import glob
import os
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_svm_metrics(test_dir, svm_model, svm_scaler,
                         suffix_filter):
    """
    Evaluate SVM on a specific test scenario and return all metrics.
    """

    all_files = glob.glob(os.path.join(test_dir, "*"))
    test_images = [
        f for f in all_files
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        and f"_{suffix_filter}" in os.path.basename(f)
    ]

    y_true = []
    y_pred = []

    print(f"\nEvaluating: {suffix_filter}")
    print(f"Images found: {len(test_images)}")

    for img_path in test_images:
        filename = os.path.basename(img_path)
        parts = filename.split('_')
        actual_pig_id = parts[1] if len(parts) >= 2 else "Unknown"

        feature_vector, _ = process_image(img_path, display=False)
        if feature_vector is None:
            continue

        X_test = svm_scaler.transform([feature_vector])
        pred_pig_id = svm_model.predict(X_test)[0]

        y_true.append(actual_pig_id)
        y_pred.append(pred_pig_id)

    metrics = {
        "Accuracy (%)": accuracy_score(y_true, y_pred) * 100,
        "Precision (%)": precision_score(y_true, y_pred, average="macro") * 100,
        "Recall (%)": recall_score(y_true, y_pred, average="macro") * 100,
        "F1-score (%)": f1_score(y_true, y_pred, average="macro") * 100,
    }

    return metrics


In [ ]:
test_dir = "/kaggle/working/augmented_testing"

scenarios = ["original", "noise", "blur", "dark", "bright"]
results = {}

for scenario in scenarios:
    results[scenario] = evaluate_svm_metrics(
        test_dir=test_dir,
        svm_model=svm_model,
        svm_scaler=svm_scaler,
        suffix_filter=scenario
    )

results_df = pd.DataFrame(results).T
results_df


In [ ]:
# ============================================================================
# CELL 12: Test SVM (sklearn) Robustness
# ============================================================================

import joblib
from PIL import Image
import glob
import cv2
import numpy as np

# Paths to your saved SVM model and scaler
svm_model_path = '/kaggle/input/all-models/tensorflow2/default/1/svm_model.pkl'
scaler_path = '/kaggle/input/all-models/tensorflow2/default/1/svm_scaler.pkl'

# Load the sklearn SVM model and scaler
svm_model = joblib.load(svm_model_path)
print("SVM model loaded successfully!")

svm_scaler = joblib.load(scaler_path)
print("Scaler loaded successfully!")

def test_svm_robustness(svm_model, svm_scaler, test_dir):
    """
    Test sklearn SVM robustness under various perturbations.
    """
    perturbations = {
        'Original': lambda x: x
    }
    
    # Get all test images
    all_files = glob.glob(os.path.join(test_dir, "*"))
    test_images = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    results = {}
    
    for pert_name, pert_func in perturbations.items():
        print(f"\nTesting SVM with {pert_name} Datasets")
        
        correct = 0
        total = 0
        
        for img_path in test_images:
            filename = os.path.basename(img_path)
            parts = filename.split('_')
            if len(parts) < 2:
                continue
            actual_pig_id = parts[1]
            
            try:
                # Read image
                img = cv2.imread(img_path)
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                # Apply perturbation
                img_pil = Image.fromarray(img_rgb)
                img_perturbed = pert_func(img_pil)
                img_perturbed = np.array(img_perturbed)
                
                # Extract features
                masked_inner_ear, filled_mask = extract_ear_region(img_perturbed)
                if masked_inner_ear is None:
                    continue
                
                final_veins, vein_features, _ = extract_vein_features(masked_inner_ear, filled_mask)
                if vein_features is None:
                    continue
                
                feature_vector = generate_feature_vector(vein_features)
                
                # Scale features and predict
                X_test = svm_scaler.transform([feature_vector])
                pred_pig_id = svm_model.predict(X_test)[0]
                
                total += 1
                if str(pred_pig_id) == actual_pig_id:
                    correct += 1
                    
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                continue
        
        accuracy = 100 * correct / total if total > 0 else 0
        results[pert_name] = accuracy
        print(f"{pert_name} Accuracy: {accuracy:.2f}% ({correct}/{total})")
    
    return results

# Run the robustness test
svm_robust_results = test_svm_robustness(svm_model, svm_scaler, test_dir)

# Save results
with open('/kaggle/working/svm_robustness.pkl', 'wb') as f:
    pickle.dump(svm_robust_results, f)

print("\nSVM robustness testing completed!")


In [ ]:
import os
from PIL import Image, ImageFilter, ImageEnhance
import numpy as np

class DataAugmentor:
    """Apply augmentations: noise, blur, brightness variations"""
    
    @staticmethod
    def add_gaussian_noise(img, std=25):
        """Add Gaussian noise"""
        img_array = np.array(img).astype(float)
        noise = np.random.normal(0, std, img_array.shape)
        noisy = np.clip(img_array + noise, 0, 255).astype(np.uint8)
        return Image.fromarray(noisy)
    
    @staticmethod
    def add_blur(img, radius=3):
        """Apply Gaussian blur"""
        return img.filter(ImageFilter.GaussianBlur(radius=radius))
    
    @staticmethod
    def decrease_brightness(img, factor=0.85):
        """Decrease brightness (-15%)"""
        enhancer = ImageEnhance.Brightness(img)
        return enhancer.enhance(factor)
    
    @staticmethod
    def increase_brightness(img, factor=1.15):
        """Increase brightness (+15%)"""
        enhancer = ImageEnhance.Brightness(img)
        return enhancer.enhance(factor)

def create_augmented_dataset(input_dir, output_dir):
    """
    Create augmented dataset: for each image, creates 5 versions
    (original, noise, blur, dark, bright)
    
    Args:
        input_dir: Directory with original images
        output_dir: Directory to save augmented images
    
    Returns:
        Number of images created
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all images
    image_files = [f for f in os.listdir(input_dir) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    print(f"\nAugmenting {len(image_files)} images...")
    print(f"Output directory: {output_dir}")
    
    total_created = 0
    
    for idx, img_filename in enumerate(image_files):
        if (idx + 1) % 50 == 0:
            print(f"  Processed {idx + 1}/{len(image_files)} images...")
        
        # Load image
        img_path = os.path.join(input_dir, img_filename)
        img = Image.open(img_path).convert('RGB')
        
        # Get base name without extension
        base_name = os.path.splitext(img_filename)[0]
        ext = os.path.splitext(img_filename)[1]
        
        # 1. Save original
        img.save(os.path.join(output_dir, f"{base_name}_original{ext}"))
        total_created += 1
        
        # 2. Gaussian noise version
        noisy = DataAugmentor.add_gaussian_noise(img, std=25)
        noisy.save(os.path.join(output_dir, f"{base_name}_noise{ext}"))
        total_created += 1
        
        # 3. Blur version
        blurred = DataAugmentor.add_blur(img, radius=3)
        blurred.save(os.path.join(output_dir, f"{base_name}_blur{ext}"))
        total_created += 1
        
        # 4. Dark version (-30% brightness)
        dark = DataAugmentor.decrease_brightness(img, factor=0.7)
        dark.save(os.path.join(output_dir, f"{base_name}_dark{ext}"))
        total_created += 1
        
        # 5. Bright version (+30% brightness)
        bright = DataAugmentor.increase_brightness(img, factor=1.3)
        bright.save(os.path.join(output_dir, f"{base_name}_bright{ext}"))
        total_created += 1
    
    print(f"\n✅ Created {total_created} augmented images")
    print(f"   Original: {len(image_files)}")
    print(f"   Augmented: {total_created} (5x multiplication)")
    
    return total_created

# ============================================================================
# RUN AUGMENTATION
# ============================================================================

# Your directories
train_dir = "/kaggle/working/pig_dataset_split/training"
test_dir = "/kaggle/working/pig_dataset_split/testing"

# Output directories
augmented_train_dir = "/kaggle/working/augmented_training"
augmented_test_dir = "/kaggle/working/augmented_testing"

print("="*80)
print("CREATING AUGMENTED DATASETS")
print("="*80)

# Augment training data
print("\n1. TRAINING DATA:")
print("-"*80)
train_count = create_augmented_dataset(train_dir, augmented_train_dir)

# Augment test data
print("\n2. TEST DATA:")
print("-"*80)
test_count = create_augmented_dataset(test_dir, augmented_test_dir)

print("\n" + "="*80)
print("AUGMENTATION COMPLETE!")
print("="*80)
print(f"\nTraining: {train_dir}")
print(f"       → {augmented_train_dir} ({train_count} images)")
print(f"\nTesting:  {test_dir}")
print(f"       → {augmented_test_dir} ({test_count} images)")
print("\n" + "="*80)
